# UCS420: Cognitive Computing — Assignment 6 (NumPy-II)

**Name:** Ali Ekram  |  **Roll No:** 1024160138  |  **Group:** 3P14

In [1]:
print("Name: Ali Ekram | Roll No: 1024160138 | Group: 3P14")
import numpy as np

Name: Ali Ekram | Roll No: 1024160138 | Group: 3P14


## Q1. Sensor readings (no for-loops)

In [2]:
temperature = np.array([25, 28, 31, 35, 38, 27, 33, 40])

# a) add the +2 °C average measurement error to every reading (vectorised)
corrected = temperature + 2
print("a) corrected      :", corrected)

# b) Celsius -> Fahrenheit:  F = 9/5 * C + 32  (applied to the original readings)
fahrenheit = temperature * 9 / 5 + 32
print("b) Fahrenheit     :", fahrenheit)

# c) Boolean indexing: readings greater than 32 °C
mask = temperature > 32
print("c) mask           :", mask)
print("   readings > 32  :", temperature[mask])

# d) count of readings exceeding 32 °C  (True counts as 1)
print("d) count > 32     :", np.count_nonzero(mask), "(same as mask.sum() =", mask.sum(), ")")

a) corrected      : [27 30 33 37 40 29 35 42]
b) Fahrenheit     : [ 77.   82.4  87.8  95.  100.4  80.6  91.4 104. ]
c) mask           : [False False False  True  True False  True  True]
   readings > 32  : [35 38 33 40]
d) count > 32     : 4 (same as mask.sum() = 4 )


**e) Why vectorisation and Boolean indexing are more efficient than looping**

* **Vectorisation** pushes the loop out of the Python interpreter and into pre-compiled C code that works on a
  contiguous block of memory. A `for` loop pays Python's per-element overhead (type checks, object creation,
  bytecode dispatch); `temperature + 2` pays it once for the whole array. The gap grows with array size
  (typically 10–100x faster), and the code is shorter and harder to get wrong.
* **Boolean indexing** expresses "which elements satisfy this condition" as one array operation
  (`temperature > 32` gives a mask, `temperature[mask]` filters). It avoids the manual
  `if` + `append` pattern, avoids building intermediate Python lists, and the mask can be reused
  (counting, filtering, replacing values) with no extra passes through Python code.

In [3]:
# quick timing demonstration
import time
big = np.random.default_rng(0).integers(20, 45, size=2_000_000)

t0 = time.perf_counter(); loop_res = [x + 2 for x in big]; t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); vec_res = big + 2;               t_vec = time.perf_counter() - t0
print(f"python loop: {t_loop:.4f}s | vectorised: {t_vec:.4f}s | speed-up ≈ {t_loop / t_vec:.0f}x")

python loop: 0.3702s | vectorised: 0.0070s | speed-up ≈ 53x


## Q2. Daily steps of four users over three days

In [4]:
steps = np.array([
    [5000, 6200, 7100],
    [8000, 7500, 9000],
    [4500, 5100, 4800],
    [9000, 8500, 9500]])

print("a) total steps          :", steps.sum())
print("b) mean steps           :", steps.mean())
print("c) max / min            :", steps.max(), "/", steps.min())
print("d) total per day (axis=0, columns = Day1..Day3):", steps.sum(axis=0))
print("e) total per user (axis=1, rows = User1..User4):", steps.sum(axis=1))

flat_idx = steps.argmax()                       # index in the flattened array
user, day = np.unravel_index(flat_idx, steps.shape)
print(f"f) argmax (flat index) = {flat_idx} -> position (user {user}, day {day}) in 0-based indexing,")
print(f"   i.e. User {user + 1}, Day {day + 1}, value = {steps[user, day]}")

a) total steps          : 84200
b) mean steps           : 7016.666666666667
c) max / min            : 9500 / 4500
d) total per day (axis=0, columns = Day1..Day3): [26500 27300 30400]
e) total per user (axis=1, rows = User1..User4): [18300 24500 14400 27000]
f) argmax (flat index) = 11 -> position (user 3, day 2) in 0-based indexing,
   i.e. User 4, Day 3, value = 9500


## Q3. Views, copies, reshaping, slicing, flatten vs ravel

In [5]:
# a) array
original = np.array([1, 2, 3, 4, 5, 6])

# b) slice index 1 to 4 (end exclusive -> indices 1,2,3) = a VIEW
subset = original[1:4]

# c) change first element of subset -> original changes too (view shares memory)
subset[0] = 999
print("c) original:", original)
print("   subset  :", subset)
print("   shares memory with original:", np.shares_memory(original, subset))

# d) slice with .copy() -> independent array
original = np.array([1, 2, 3, 4, 5, 6])          # reset so the demo starts clean
copied = original[1:4].copy()
copied[0] = 500
print("\nd) original:", original)
print("   copied  :", copied, "(original unaffected)")

c) original: [  1 999   3   4   5   6]
   subset  : [999   3   4]
   shares memory with original: True

d) original: [1 2 3 4 5 6]
   copied  : [500   3   4] (original unaffected)


In [6]:
# e) 1..12 as a 3x4 matrix
m = np.arange(1, 13).reshape(3, 4)
print("e) matrix:\n", m)

# f) indexing / slicing  (rows and columns counted from 1 in the question)
print("\nf) first row            :", m[0])
print("   last row             :", m[-1])
print("   second column        :", m[:, 1])
print("   rows 1-2, columns 2-3:\n", m[0:2, 1:3])

e) matrix:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

f) first row            : [1 2 3 4]
   last row             : [ 9 10 11 12]
   second column        : [ 2  6 10]
   rows 1-2, columns 2-3:
 [[2 3]
 [6 7]]


In [7]:
# g) flatten() vs ravel()
flat_copy = m.flatten()      # always returns a COPY
flat_view = m.ravel()        # returns a VIEW when possible (here: contiguous array -> view)
print("g) flatten():", flat_copy)
print("   ravel()  :", flat_view)

# h) modify ravel() result -> original matrix changes
flat_view[0] = 100
print("\nh) after ravel()[0] = 100  -> original matrix:\n", m)
print("   original affected?", m[0, 0] == 100)

# i) modify flatten() result -> original is not changed
m = np.arange(1, 13).reshape(3, 4)       # reset
flat_copy = m.flatten()
flat_copy[0] = 200
print("\ni) after flatten()[0] = 200 -> original matrix:\n", m)
print("   original affected?", m[0, 0] == 200)

# j) attributes
print("\nj) shape:", m.shape, "| ndim:", m.ndim, "| size:", m.size, "| dtype:", m.dtype)

g) flatten(): [ 1  2  3  4  5  6  7  8  9 10 11 12]
   ravel()  : [ 1  2  3  4  5  6  7  8  9 10 11 12]

h) after ravel()[0] = 100  -> original matrix:
 [[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]
   original affected? True

i) after flatten()[0] = 200 -> original matrix:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
   original affected? False

j) shape: (3, 4) | ndim: 2 | size: 12 | dtype: int64


**Observation:** `ravel()` normally returns a *view* (it only copies when the memory layout forces it),
so changing it changes the original. `flatten()` always returns a *copy*, so the original is safe.

## Q4. Assistance score prediction with Ordinary Least Squares

In [8]:
X = np.array([
    [6, 70, 3],
    [5, 50, 6],
    [8, 80, 2],
    [4, 30, 8]
])
y = np.array([40, 65, 30, 85])

# a) shape and dimensions
print("a) X.shape =", X.shape, "| X.ndim =", X.ndim, "-> 4 users (rows) x 3 features (columns)")

# b) transpose
print("\nb) X.T =\n", X.T, "\n   shape:", X.T.shape)

a) X.shape = (4, 3) | X.ndim = 2 -> 4 users (rows) x 3 features (columns)

b) X.T =
 [[ 6  5  8  4]
 [70 50 80 30]
 [ 3  6  2  8]] 
   shape: (3, 4)


**Transpose:** `X.T` swaps rows and columns. Here `X` has one *user* per row; `X.T` has one *feature* per row
(row 0 = sleep hours of all four users, row 1 = activity, row 2 = stress). It is what lets `X.T @ X`
combine every feature with every other feature across all users.

In [9]:
# c) X.T @ X  (3x3 matrix of sums of products between features)
XtX = X.T @ X
print("c) X.T @ X =\n", XtX)

# d) inverse (only valid if the matrix is non-singular)
print("\n   det(X.T @ X) =", np.linalg.det(XtX))
XtX_inv = np.linalg.inv(XtX)
print("d) inv(X.T @ X) =\n", XtX_inv)
print("   check  XtX @ inv ≈ I ?", np.allclose(XtX @ XtX_inv, np.eye(3)))

# e) OLS:  w = (XᵀX)⁻¹ Xᵀ y
w = XtX_inv @ X.T @ y
print("\ne) coefficients w =", w)
print("   cross-check with np.linalg.lstsq:", np.linalg.lstsq(X, y, rcond=None)[0])
print("   fitted values X @ w =", X @ w, "| actual y =", y)

c) X.T @ X =
 [[  141  1430    96]
 [ 1430 14700   910]
 [   96   910   113]]

   det(X.T @ X) = 753700.000000011
d) inv(X.T @ X) =
 [[ 1.10521428 -0.09848746 -0.14581398]
 [-0.09848746  0.00891203  0.01190129]
 [-0.14581398  0.01190129  0.0368847 ]]
   check  XtX @ inv ≈ I ? True

e) coefficients w = [-0.15921454  0.14083853 10.06302242]
   cross-check with np.linalg.lstsq: [-0.15921454  0.14083853 10.06302242]
   fitted values X @ w = [39.09247711 66.62398832 30.11941091 84.09247711] | actual y = [40 65 30 85]


**f) Meaning of the coefficients** — each coefficient is the change in the predicted assistance score for a
one-unit increase in that feature while the other two features stay fixed:

* `w[0] ≈ -0.16` → **sleep hours**: each extra hour of sleep lowers the predicted assistance score by about
  0.16 points (small effect, negative direction: more sleep → slightly less assistance needed).
* `w[1] ≈ +0.14` → **activity level**: each extra activity point raises the predicted score by about 0.14.
  This is counter-intuitive; with only 4 users, activity, sleep and stress are strongly correlated
  (collinear), so the individual coefficient is unreliable.
* `w[2] ≈ +10.06` → **stress level**: each extra stress point raises the predicted score by about 10 points —
  by far the dominant factor (more stress → much more assistance needed).

The sign tells the direction of the relationship, the magnitude its strength *in the units of that feature*
(so coefficients are not directly comparable unless the features are on the same scale).
Note: the model has no intercept column (exactly as the given formula), so it assumes a score of 0 when all
features are 0. With only 4 samples the fit is also very sensitive to the data — treat it as a learning exercise.

In [10]:
# g) prediction for a new user
new_user = np.array([5, 40, 7])
prediction = new_user @ w
print("g) predicted assistance score for", new_user, "=", round(float(prediction), 3))

g) predicted assistance score for [ 5 40  7] = 75.279
